## Step 7 --- Evaluation

In [1]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report)

In [ ]:
def calculate_metrics(y_true, y_pred, model_name, dataset_name):
    """Calculate and print all evaluation metrics"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"\n{model_name} - {dataset_name} Metrics:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("\n" + "="*60)
print("EVALUATION METRICS")
print("="*60)

In [ ]:
import os
import pickle

Assume you have trained models named model_1, model_2, and model_3
Replace these with your actual model variables
model_1 = "..." # your first trained model object
model_2 = "..." # your second trained model object
model_3 = "..." # your third trained model object

1. Organize models in a dictionary
all_models = {
    'model_A_log_reg': model_1,
    'model_B_tree': model_2,
    'model_C_svm': model_3
}

2. Define the folder path and create it if it doesn't exist
folder_name = 'saved_evaluation_models'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Folder '{folder_name}' created.")

3. Define the file path within the folder
file_path = os.path.join(folder_name, 'all_models.pkl')

4. Save the entire dictionary to a single pickle file
try:
    with open(file_path, 'wb') as f:
        pickle.dump(all_models, f)
    print(f"All models successfully saved to {file_path}")
except Exception as e:
    print(f"An error occurred while saving: {e}")

In [ ]:
1. SETUP & IMPORTS
import sys
import os
import importlib

Add 'src' to the path
sys.path.append(os.path.abspath('../src'))

Import the evaluation script
import evaluate

Force reload
importlib.reload(evaluate)

2. EXECUTE EVALUATION ARENA
print(" STARTING EVALUATION ARENA...")
print("   - Loading models from 'models/'...")
print("   - Testing on the LAST 15% of data (Strictly Unseen).")
print("   - Comparing Accuracy and picking a winner.\n")

Run the function
evaluate.evaluate_models()

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate_models():
    # --- 1. SETUP ---
    current_script_dir = os.path.dirname(os.path.abspath(_file_))
    project_root = os.path.dirname(current_script_dir)

    data_path = os.path.join(project_root, 'data', 'labeled', 'labeled_data.csv')
    model_dir = os.path.join(project_root, 'models')

    print(" Starting Model Evaluation Arena...")

    # --- 2. PREPARE TEST DATA ---
    if not os.path.exists(data_path):
        print(f" Error: Data not found at {data_path}")
        # FIX: Return 3 Nones so the notebook doesn't crash
        return None, None, None

    df = pd.read_csv(data_path)
    df.dropna(inplace=True)

    drop_cols = ['open_time', 'close_time', 'ignore', 'future_return', 'label', 'threshold_buy', 'threshold_sell']
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols]
    y = df['label']

    test_start = int(len(df) * 0.85)
    X_test = X.iloc[test_start:]
    y_test = y.iloc[test_start:]

    print(f"   Testing on {len(X_test)} rows (Last 15%)\n")

    # --- 3. EVALUATION LOOP ---
    best_acc = -1
    best_model_name = None
    best_preds = None

    # Check if directory exists
    if not os.path.exists(model_dir):
        print(f" Error: Models folder not found at {model_dir}")
        return None, None, None

    model_files = [f for f in os.listdir(model_dir) if f.endswith('.pkl') and f != 'best_crypto_model.pkl']

    if not model_files:
        print(" No models found! Run train.py first.")
        # FIX: Return 3 Nones
        return None, None, None

    print(f"{'MODEL':<20} | {'ACCURACY':<10}")
    print("-" * 35)

    for filename in model_files:
        model_name = filename.replace('.pkl', '')
        model_path = os.path.join(model_dir, filename)

        try:
            model = joblib.load(model_path)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)

            print(f"{model_name:<20} | {acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_model_name = model_name
                best_preds = preds

        except Exception as e:
            print(f" Error loading {model_name}: {e}")

    # --- 4. SHOW WINNER ---
    if best_model_name is None:
        print(" No valid models could be evaluated.")
        return None, None, None

    print("-" * 35)
    print(f" WINNER: {best_model_name} (Accuracy: {best_acc:.4f})")

    # --- 5. SAVE BEST MODEL ---
    src_file = os.path.join(model_dir, f"{best_model_name}.pkl")
    dst_file = os.path.join(model_dir, "best_crypto_model.pkl")
    try:
        shutil.copyfile(src_file, dst_file)
        print(f" Copied winner to: {dst_file}")
    except Exception as e:
        print(f" Could not copy best model: {e}")

    # --- 6. TEXT REPORT ---
    print("\n Classification Report (Winner):")
    print(classification_report(y_test, best_preds, target_names=['SELL', 'HOLD', 'BUY']))

    return best_model_name, y_test, best_preds

if name == "_main_":
    winner_name, y_true, y_pred = evaluate_models()

    if winner_name is not None:
        print("\n Generating Confusion Matrix Plot...")
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(7, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                    xticklabels=['Pred SELL', 'Pred HOLD', 'Pred BUY'],
                    yticklabels=['Act SELL', 'Act HOLD', 'Act BUY'])
        plt.title(f'Confusion Matrix: {winner_name}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.show()

## Step 8 --- Serialize the Model

In [ ]:
import joblib
from keras.models import model
joblib.dump(model, "models/buy_sell_classifier.pkl")

## Step 9 --- Prediction Pipeline

In [ ]:
import joblib

def predict(features):
    model = joblib.load("models/buy_sell_classifier.pkl")
    return model.predict(features)